# 原39项直接预测 SOXX 桥接优势的固定信息检验

先核对旧Qlib双任务标签与预测逐日一致，再只拟合一次直接联合接收器；所有预测、模型和重采样细节保存在新Recorder及NAS。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_original39_joint_soxx_bridge_information_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["budget"]["new_market_GETs"]==plan["budget"]["new_native_accounts"]==0
parent=root/src["parent_card"];dual=root/src["dual_card"]
assert sha(parent/"summary.json")==src["parent_summary_sha256"]
assert sha(dual/"summary.json")==src["dual_summary_sha256"]
assert sha(root/"src/analysis/seed_disagreement_information.py")==src["bootstrap_module_sha256"]
ps=json.loads((parent/"summary.json").read_text());ds=json.loads((dual/"summary.json").read_text())
assert ds["nas_evidence"]["sha256"]==src["dual_NAS_sha256"] and ds["nas_evidence"]["size_bytes"]==src["dual_NAS_size_bytes"]
assert ds["nas_readback"]["verified"] and ds["nas_readback"]["sha256"]==src["dual_NAS_sha256"]
base=ps["input_audit"]["original_features"]
assert len(base)==39
artifacts=dual/"qlib_recorder"/src["dual_experiment_id"]/src["dual_recorder_id"]/"artifacts"
assert sha(artifacts/"inputs/dataset")==src["dataset_artifact_sha256"]
assert sha(artifacts/"predictions/scored")==src["scored_artifact_sha256"]
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((dual/"qlib_recorder").resolve())):
 rec=R.get_recorder(experiment_id=src["dual_experiment_id"],recorder_id=src["dual_recorder_id"])
 dataset=rec.load_object("inputs/dataset")
 old=rec.load_object("predictions/scored")
train=dataset.prepare("train",col_set=["feature","label","meta"])
held=dataset.prepare("held",col_set=["feature","label","meta"])
train=train.loc[train[("meta","train_mature")].astype(bool)]
held=held.loc[held[("meta","held_mature")].astype(bool)]
assert len(train)==plan["cohorts"]["expected_train_mature_rows"]==3014
assert len(held)==len(old)==plan["cohorts"]["expected_held_mature_rows"]==940
assert list(train["feature"].columns)==list(held["feature"].columns)==base
train_dates=pd.DatetimeIndex(train.index.get_level_values("datetime"));held_dates=pd.DatetimeIndex(held.index.get_level_values("datetime"))
assert train_dates.is_monotonic_increasing and held_dates.is_monotonic_increasing
assert held_dates.equals(pd.DatetimeIndex(old.signal_date))
for dataset_name,pred_name in [("cash_advantage_H5","cash_target"),("SOXL_relative_H5","SOXL_relative_target")]:
 assert np.allclose(held[("label",dataset_name)].to_numpy(float),old[pred_name].to_numpy(float),rtol=0,atol=1e-12)
calendar=pd.DatetimeIndex(D.calendar(start_time="2023-01-03",end_time="2026-10-07",freq="day"))
assert len(calendar)==plan["cohorts"]["held_full_calendar_rows"]==944 and held_dates.isin(calendar).all()
ytr=np.minimum(-train[("label","cash_advantage_H5")].to_numpy(float),-train[("label","SOXL_relative_H5")].to_numpy(float))
yhe=np.minimum(-held[("label","cash_advantage_H5")].to_numpy(float),-held[("label","SOXL_relative_H5")].to_numpy(float))
old_control=np.minimum(-old.cash_prediction.to_numpy(float),-old.SOXL_relative_prediction.to_numpy(float))
assert np.isfinite(ytr).all() and np.isfinite(yhe).all() and np.isfinite(old_control).all()
assert float(np.std(ytr))>0
preflight={"status":"joint_target_source_date_maturity_and_control_parity_pass","plan_sha256":sha(plan_path),
           "source_recorder_id":src["dual_recorder_id"],"feature_columns":len(base),"train_mature_rows":len(train),
           "held_mature_rows":len(held),"full_held_calendar_rows":len(calendar),"saved_two_head_targets_equal":True,
           "new_model_fits":0,"new_native_accounts":0}
(card/"preflight_result.json").write_text(json.dumps(preflight,ensure_ascii=False,indent=2)+"\n")
print(preflight)


{'status': 'joint_target_source_date_maturity_and_control_parity_pass', 'plan_sha256': 'c508fac456f2e37dc8bbd992317a9349fefd34aaa0099a7ac1cb50ca61d9dd97', 'source_recorder_id': 'a4bbc11f5c764ac38ff2f682eadd0e1d', 'feature_columns': 39, 'train_mature_rows': 3014, 'held_mature_rows': 940, 'full_held_calendar_rows': 944, 'saved_two_head_targets_equal': True, 'new_model_fits': 0, 'new_native_accounts': 0}


In [2]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence
import tempfile
assert preflight["status"]=="joint_target_source_date_maturity_and_control_parity_pass"
state_path=card/"run_state.json"
if state_path.exists():raise ValueError("fixed direct joint-receiver fit already attempted; do not repeat")
state={"status":"fit_attempted","plan_sha256":sha(plan_path),"new_model_fits":0,"bootstrap_attempts":0,"new_native_accounts":0}
state_path.write_text(json.dumps(state,indent=2)+"\n")
model=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),Ridge(alpha=1.0))
model.fit(train["feature"][base],ytr)
state["new_model_fits"]=1;state_path.write_text(json.dumps(state,indent=2)+"\n")
pred=model.predict(held["feature"][base])
mean_control=float(np.mean(ytr))
delta=(yhe-old_control)**2-(yhe-pred)**2
mean_delta=(yhe-mean_control)**2-(yhe-pred)**2
score=pd.DataFrame({"signal_date":held_dates,"target":yhe,"saved_two_head_control":old_control,
                    "candidate":pred,"paired_improvement":delta,"improvement_vs_train_mean":mean_delta})
metrics={"MSE_saved_two_head_control":float(np.mean((yhe-old_control)**2)),
         "MSE_candidate":float(np.mean((yhe-pred)**2)),"MSE_train_mean":float(np.mean((yhe-mean_control)**2)),
         "paired_improvement":float(np.mean(delta)),"improvement_vs_train_mean":float(np.mean(mean_delta))}
metrics["held_halves"]=[]
for start,end in plan["cohorts"]["held_halves"]:
 mask=score.signal_date.between(pd.Timestamp(start),pd.Timestamp(end))
 metrics["held_halves"].append({"start":start,"end":end,"rows":int(mask.sum()),"paired_improvement":float(score.loc[mask,"paired_improvement"].mean())})
metrics["complete_years"]={str(year):{"rows":int(score.signal_date.dt.year.eq(year).sum()),
    "paired_improvement":float(score.loc[score.signal_date.dt.year.eq(year),"paired_improvement"].mean())} for year in plan["cohorts"]["complete_held_years"]}
full=pd.Series(np.nan,index=calendar,dtype=float);full.loc[held_dates]=delta
assert int(full.notna().sum())==940
fixed=plan["information_gate"]["full_944_day_stationary_block_lower_positive"]
boot,_,means=full_calendar_bootstrap(full.to_numpy(float),mean_block=fixed["mean_block_sessions"],
    resamples=fixed["replicates"],seed=fixed["seed"],alpha=fixed["one_sided_alpha"])
state["bootstrap_attempts"]=1;state_path.write_text(json.dumps(state,indent=2)+"\n")
gates={"beats_both_controls":metrics["MSE_candidate"]<metrics["MSE_saved_two_head_control"] and metrics["MSE_candidate"]<metrics["MSE_train_mean"],
       "both_halves_positive":all(x["paired_improvement"]>0 for x in metrics["held_halves"]),
       "both_complete_years_positive":all(x["paired_improvement"]>0 for x in metrics["complete_years"].values()),
       "block_lower_positive":boot["lower_bound"]>0}
decision="conditional_direct_joint_margin_information_pass_needs_new_f696_account" if all(gates.values()) else "not_promoted_fixed_original39_joint_margin_information_failed"
with R.start(experiment_name=card.name,uri=str((card/"qlib_recorder").resolve())):
 recorder=R.get_recorder()
 recorder.save_objects(artifact_path="inputs",plan=plan,preflight=preflight,dataset=dataset)
 recorder.save_objects(artifact_path="models",joint_margin=model)
 recorder.save_objects(artifact_path="predictions",held_scored=score)
 recorder.save_objects(artifact_path="analysis",metrics=metrics,gates=gates,bootstrap=boot,
                       full_held_calendar=full,bootstrap_replicate_means=means)
 R.log_params(receiver="direct_joint_SOXX_over_cash_and_SOXL_H5",parent="f696_plus_b472",inputs="original39")
 R.log_metrics(eval_paired_MSE_improvement=metrics["paired_improvement"],block_lower=boot["lower_bound"],
               held_mature_rows=len(held))
 rid,eid=recorder.id,str(recorder.experiment_id)
state.update(status="fitted_and_recorded",recorder_id=rid,experiment_id=eid)
state_path.write_text(json.dumps(state,indent=2)+"\n")
parent_binding=json.loads((root/"research/minute_information_pit_allocator_20261012/nas_binding.json").read_text())
binding={"schema_version":"joint_SOXX_margin_recorder_backup_v1","authority":"yanxuan_nas","ssh":parent_binding["ssh"],
         "source_id":"original39_joint_margin_"+sha(plan_path)[:16]}
binding_path=card/"nas_binding.json";binding_path.write_text(json.dumps(binding,indent=2)+"\n")
nas=None;nas_error=None;readback=False
try:
 nas=archive_nas_recorder(binding_path,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
 with tempfile.TemporaryDirectory(prefix="joint-margin-readback-") as temp:
  restored=Path(temp);materialize_nas_evidence(binding_path,card=restored)
  with R.uri_context(str((restored/"qlib_recorder").resolve())):
   check=R.get_recorder(experiment_id=eid,recorder_id=rid)
   saved_metrics=check.load_object("analysis/metrics")
   saved_gates=check.load_object("analysis/gates")
  readback=saved_metrics==metrics and saved_gates==gates
except Exception as exc:nas_error=f"{type(exc).__name__}: {str(exc)[:300]}"
result={"status":decision if readback else "needs_NAS_archive_or_readback_after_fixed_fit",
        "scientific_decision":decision,"objective_achieved":False,"plan_sha256":sha(plan_path),
        "preflight":"preflight_result.json","metrics":metrics,"gates":gates,"bootstrap":boot,
        "recorder_id":rid,"experiment_id":eid,"NAS_evidence":nas,"NAS_readback_verified":readback,"NAS_error":nas_error,
        "new_model_fits":1,"new_bootstraps":1,"new_native_accounts":0,"independent_forward_days":0,
        "historical_2023_2026_reused":True,"SOXX_goal_achieved":False,
        "next_action":"Close the fixed information receiver on any gate failure. If all gates pass and NAS readback is verified, separately freeze an exact-f696 own-state 0bp Tiered mapping and full account; this result alone cannot promote."}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
state["status"]="completed" if readback else "needs_NAS_readback";state_path.write_text(json.dumps(state,indent=2)+"\n")
print({"status":result["status"],"MSE":{k:v for k,v in metrics.items() if k.startswith("MSE")},
       "paired_improvement":metrics["paired_improvement"],"block_lower":boot["lower_bound"],"gates":gates,"NAS_readback":readback})


{'status': 'conditional_direct_joint_margin_information_pass_needs_new_f696_account', 'MSE': {'MSE_saved_two_head_control': 0.004860988106974056, 'MSE_candidate': 0.0027607193108684642, 'MSE_train_mean': 0.0028267215008935644}, 'paired_improvement': 0.0021002687961055914, 'block_lower': 0.001388795245415948, 'gates': {'beats_both_controls': True, 'both_halves_positive': True, 'both_complete_years_positive': True, 'block_lower_positive': True}, 'NAS_readback': True}


## 中文结论

在封存的Qlib DatasetH中，将现金相对SOXX H5优势与SOXL相对SOXX H5优势合成一个新接收任务：两者取负后的较小值，表示**SOXX同时胜过现金和SOXL的较小毛价格优势**。旧两个Ridge头的预测先取相同的最小值作同日对照；新版本只用原39项输入按固定 `SimpleImputer→StandardScaler→Ridge(alpha=1)` 拟合一次。来源哈希、3014条成熟训练样本、940条成熟复用评价样本、两旧目标逐日相等及完整944日日历均先通过预检。

评价MSE：旧两头组合 **0.00486099**，直接联合头 **0.00276072**，联合训练均值 **0.00282672**。候选相对旧两头的配对MSE改善 **+0.00210027**，2023–2024／2025–2026两半及完整2024、2025年均为正；冻结的944日历、63日平均区块、10000次校正重采样下界 **+0.00138880**。四项固定信息门均通过，NAS Recorder及指标／门回读也通过。

这个结果**仅允许进入下一项独立冻结的经济检验**：相对训练均值的整体改善只有 **0.00006600**，本次校正区块门比较的是表现不佳的旧两头组合，不能把它称为SOXX超额。标签是未扣Tiered费用、没有自身风险反馈的H5价格边际；2023–2026历史已复用，尚无新0bp资金账户、2/5bp或合格前向日。下一步必须先冻结分数如何在精确f696五输出与b472动作／风险层中转移现金或SOXL到SOXX，核对944/944母本身份，再跑**候选自身完整0bp Tiered账户**及全部年／季度／126日和统计风险门。条件性决定 `conditional_direct_joint_margin_information_pass_needs_new_f696_account`，SOXX目标仍未达成。
